In [1]:
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from transformers import CamembertTokenizer, CamembertModel
import torch
from tqdm import tqdm

/users/eleves-b/2023/francois.loning/Documents/kaggle-challenge/kaggle-env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Data loading and exploration

In [2]:
TEXT_COL = "full_text"
ID_COL = "challenge_id"
DATA_DIR = "./data"
EMB_DIR = os.path.join(DATA_DIR, "embeddings")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 16

# 1. Load JSONL files
# ============================

train = pd.read_json("data/train.jsonl", lines=True)
train = json_normalize(train.to_dict(orient="records"))

X_kaggle = pd.read_json("data/kaggle_test.jsonl", lines=True)
X_kaggle = json_normalize(X_kaggle.to_dict(orient="records"))

X_train = train.drop("label", axis=1)
y_train = train["label"]
np.save(os.path.join(DATA_DIR, "y_train.npy"), y_train)


In [3]:
# 2. Extract text
# ============================

def extract_full_text(row):
    txt = row["text"]
    if not pd.isna(row.get("extended_tweet.full_text", np.nan)):
        txt = row["extended_tweet.full_text"]
    return txt

X_train[TEXT_COL] = X_train.apply(extract_full_text, axis=1)
X_kaggle[TEXT_COL] = X_kaggle.apply(extract_full_text, axis=1)

# 2. Data pre-processing

In [5]:
# 3. Drop list/dict columns
# ============================

def drop_list_columns(df):
    bad = []
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, (list, dict))).any():
            bad.append(col)
    return df.drop(columns=bad)

X_train_clean = drop_list_columns(X_train)
X_kaggle_clean = drop_list_columns(X_kaggle)

# ============================
# 4. Synchronize columns
# ============================

common_cols = list(set(X_train_clean.columns) & set(X_kaggle_clean.columns))

if ID_COL not in common_cols:
    print("WARNING: ID column missing in one dataset!")

# Keep text column separate
if TEXT_COL in common_cols:
    common_cols.remove(TEXT_COL)

# Final column lists
X_train_clean = X_train_clean[[TEXT_COL] + common_cols]
X_kaggle_clean = X_kaggle_clean[[TEXT_COL] + common_cols]

In [7]:
# 5. Remove columns with NaN in train OR kaggle
# ============================

nan_train = X_train_clean.isna().any()
nan_kaggle = X_kaggle_clean.isna().any()

to_drop = nan_train[nan_train].index.union(nan_kaggle[nan_kaggle].index).tolist()

# Never drop TEXT or ID
for c in [TEXT_COL, ID_COL]:
    if c in to_drop:
        to_drop.remove(c)

X_train_proc = X_train_clean.drop(columns=to_drop)
X_kaggle_proc = X_kaggle_clean.drop(columns=to_drop)

# 6. Structured columns
# ============================

structured_cols = [c for c in X_train_proc.columns if c not in [TEXT_COL, ID_COL]]

numeric_cols = X_train_proc[structured_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train_proc[structured_cols].select_dtypes(include=["object", "bool"]).columns.tolist()


In [8]:
# 7. Preprocess structured features
# ============================

preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
    ],
    remainder="drop"
)

X_train_struct = preprocessor.fit_transform(X_train_proc.drop(columns=[TEXT_COL, ID_COL]))
X_kaggle_struct = preprocessor.transform(X_kaggle_proc.drop(columns=[TEXT_COL, ID_COL]))

# ============================
# 8. Extract ID arrays
# ============================

train_ids = X_train_proc[ID_COL].astype(float).values.reshape(-1, 1)
kaggle_ids = X_kaggle_proc[ID_COL].astype(float).values.reshape(-1, 1)


In [9]:
embedding_dir = os.path.join(DATA_DIR, "embeddings")

In [ ]:
# 9. EMBED TEXT USING CAMEMBERT
# ============================
os.makedirs(embedding_dir, exist_ok=True)
train_embedding_path = os.path.join(embedding_dir, "X_train_text_embeddings.npy")
kaggle_embedding_path = os.path.join(embedding_dir, "X_kaggle_text_embeddings.npy")
# -----------------------------------------------------------------

tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
model = CamembertModel.from_pretrained('camembert-base')
model.to(device)
model.eval()

def embed_texts(texts, batch_size=16):
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            # Use [CLS] token representation (first token) as embedding
            batch_embeddings = outputs.last_hidden_state[:,0,:].cpu().numpy()
            embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

def embed_texts_multilayer(texts, batch_size=16, layers=[6, 9, 12]):
    """Extract embeddings from multiple transformer layers for richer representations."""
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i+batch_size]
            encoded = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            # Get hidden states from all layers
            outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
            
            # Extract [CLS] token from specified layers and concatenate
            layer_embeddings = []
            for layer_idx in layers:
                layer_emb = outputs.hidden_states[layer_idx][:, 0, :].cpu().numpy()
                layer_embeddings.append(layer_emb)
            
            # Concatenate horizontally: 3 layers × 768 = 2304 dimensions
            batch_embeddings = np.hstack(layer_embeddings)
            embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

# Extract raw texts
train_texts = X_train_proc[TEXT_COL].astype(str).tolist()
kaggle_texts = X_kaggle_proc[TEXT_COL].astype(str).tolist()

# Compute the embeddings
print("Embedding TRAIN texts...")
train_embeddings = embed_texts(train_texts, batch_size=batch_size)
np.save(train_embedding_path, train_embeddings)
print("Train embeddings saved to:", train_embedding_path)

print("Embedding KAGGLE texts...")
kaggle_embeddings = embed_texts(kaggle_texts, batch_size=batch_size)
np.save(kaggle_embedding_path, kaggle_embeddings)
print("Kaggle embeddings saved to:", kaggle_embedding_path)

# ============================
# 10. MULTI-LAYER EMBEDDINGS
# ============================
print("\n" + "="*70)
print("EXTRACTING MULTI-LAYER EMBEDDINGS")
print("="*70)
print("Extracting from layers 6, 9, and 12 for richer representations...")

train_multilayer_path = os.path.join(embedding_dir, "X_train_multilayer_embeddings.npy")
kaggle_multilayer_path = os.path.join(embedding_dir, "X_kaggle_multilayer_embeddings.npy")

print("\nEmbedding TRAIN texts (multi-layer)...")
train_multilayer_embeddings = embed_texts_multilayer(train_texts, batch_size=batch_size, layers=[6, 9, 12])
np.save(train_multilayer_path, train_multilayer_embeddings)
print(f"Train multi-layer embeddings saved: {train_multilayer_embeddings.shape}")
print(f"Saved to: {train_multilayer_path}")

print("\nEmbedding KAGGLE texts (multi-layer)...")
kaggle_multilayer_embeddings = embed_texts_multilayer(kaggle_texts, batch_size=batch_size, layers=[6, 9, 12])
np.save(kaggle_multilayer_path, kaggle_multilayer_embeddings)
print(f"Kaggle multi-layer embeddings saved: {kaggle_multilayer_embeddings.shape}")
print(f"Saved to: {kaggle_multilayer_path}")

print("\n" + "="*70)
print("Multi-layer embeddings extraction complete!")
print(f"Single-layer: 768 dimensions")
print(f"Multi-layer: {train_multilayer_embeddings.shape[1]} dimensions (3 × 768)")
print("="*70)


Embedding TRAIN texts...


100%|██████████| 9683/9683 [04:24<00:00, 36.64it/s]


Train embeddings saved to: ./data/embeddings/X_train_text_embeddings.npy
Embedding KAGGLE texts...


100%|██████████| 6462/6462 [02:56<00:00, 36.61it/s]


Kaggle embeddings saved to: ./data/embeddings/X_kaggle_text_embeddings.npy


In [12]:
print(X_train_struct.shape)
X_train_text = np.load(train_embedding_path)
print(X_train_text.shape)

(154914, 8)
(154914, 768)


In [ ]:
train_embedding_path = os.path.join(embedding_dir, "X_train_text_embeddings.npy")
kaggle_embedding_path = os.path.join(embedding_dir, "X_kaggle_text_embeddings.npy")

X_train_text = np.load(train_embedding_path)
X_kaggle_text = np.load(kaggle_embedding_path)

print("Loaded text embeddings:")
print("train:", X_train_text.shape)
print("kaggle:", X_kaggle_text.shape)


train_full_path = os.path.join(DATA_DIR, "X_train_processed.npy")
kaggle_full_path = os.path.join(DATA_DIR, "X_kaggle_processed.npy")


print("Concatenating structured features and text embeddings...")

# Horizontal concatenation
X_train_full = np.hstack([X_train_struct, X_train_text])
X_kaggle_full = np.hstack([X_kaggle_struct, X_kaggle_text])

# Save processed arrays
np.save(train_full_path, X_train_full)
np.save(kaggle_full_path, X_kaggle_full)
print("Final feature arrays saved.")

# ============================
# 11. MULTI-LAYER CONCATENATION
# ============================
print("\n" + "="*70)
print("CREATING MULTI-LAYER FEATURE ARRAYS")
print("="*70)

train_multilayer_path = os.path.join(embedding_dir, "X_train_multilayer_embeddings.npy")
kaggle_multilayer_path = os.path.join(embedding_dir, "X_kaggle_multilayer_embeddings.npy")

X_train_multilayer = np.load(train_multilayer_path)
X_kaggle_multilayer = np.load(kaggle_multilayer_path)

print("Loaded multi-layer embeddings:")
print("  train:", X_train_multilayer.shape)
print("  kaggle:", X_kaggle_multilayer.shape)

train_multilayer_full_path = os.path.join(DATA_DIR, "X_train_processed_multilayer.npy")
kaggle_multilayer_full_path = os.path.join(DATA_DIR, "X_kaggle_processed_multilayer.npy")

print("\nConcatenating structured features (8) + multi-layer embeddings (2304)...")

# Horizontal concatenation: 8 + 2304 = 2312 total features
X_train_multilayer_full = np.hstack([X_train_struct, X_train_multilayer])
X_kaggle_multilayer_full = np.hstack([X_kaggle_struct, X_kaggle_multilayer])

# Save processed arrays
np.save(train_multilayer_full_path, X_train_multilayer_full)
np.save(kaggle_multilayer_full_path, X_kaggle_multilayer_full)

print(f"\nMulti-layer feature arrays saved:")
print(f"  Train shape: {X_train_multilayer_full.shape}")
print(f"  Kaggle shape: {X_kaggle_multilayer_full.shape}")
print(f"  Saved to: {train_multilayer_full_path}")
print(f"  Saved to: {kaggle_multilayer_full_path}")
print("="*70)


Loaded text embeddings:
train: (154914, 768)
kaggle: (103380, 768)
Concatenating structured features and text embeddings...
Final feature arrays saved.


In [15]:
X_train_struct.head()

AttributeError: 'numpy.ndarray' object has no attribute 'head'